# Validate LFP and first-order extracellular field

People need contact-free electrodes instead of patch clamps, and we need to help them.  
Is our modelling precise, or at least consistent?

Let's see with this simple example from Brian's cookbook: https://brian2.readthedocs.io/en/stable/examples/compartmental.lfp.html

In [ ]:
import numpy as np
cable_length_microns = 100_000
cable_diameter_microns = 2*238
cable_compartments = 1000
extracell_conductivity_mho_per_m = 0.3 # siemens/meter

## NEURON model and result
Let's see how [Neuron]( https://web.archive.org/web/20100730180342/http://neuron.yale.edu/phpBB/viewtopic.php?f=28&t=168 ) sees it: 

In [ ]:
!wget https://web.archive.org/web/20100730180342/http://www.neuron.yale.edu/ftp/ted/neuron/extracellular_stim_and_rec.zip 
!unzip -xoq extracellular_stim_and_rec.zip 

In [ ]:
!cd extracellular_stim_and_rec/ && nrnivmodl

In [ ]:
import os; basedir = os.getcwd()

In [ ]:
os.chdir(basedir + '/extracellular_stim_and_rec')

In [ ]:
from neuron import h; import neuron.units

In [ ]:
for sec in h.allsec():
    # print(sec)
    h.delete_section(sec=sec)

In [ ]:
h.load_file("stdrun.hoc")

In [ ]:
# Morphology
cable = h.Section(name="cable")
cable.L    = cable_length_microns   * neuron.units.μm
cable.diam = cable_diameter_microns * neuron.units.μm
# h(f" create cable {{ cable {{ pt3dclear() pt3dadd(0, 0, 0, {cable_diameter_microns}) pt3dadd(0, 0, {cable_length_microns}, {cable_diameter_microns}) }} }}"); cable = h.cable
cable.nseg = cable_compartments
# Cable biophysics
cable.Ra = 35.4  # Axial resistance in Ohm * cm
cable.cm = 1  # Membrane capacitance in micro Farads / cm^2
# HH mech
cable.insert(h.hh)
for seg in cable: 
    seg.hh.gnabar =  0.12 if seg.x < .5 else 0  # Sodium conductance in S/cm2 
    seg.hh.gkbar = 0.036  # Potassium conductance in S/cm2
    seg.hh.gl = 0.0003  # Leak conductance in S/cm2
    seg.hh.el = -54.387 * neuron.units.mV

In [ ]:
# Add the mechs, the following loops will look for them
cable.insert('extracellular')
for seg in cable:
    for idx in range(2):
        seg.xraxial[idx] = 1e+09
        seg.xg[idx]=1e+09
        seg.xc[idx]=0
        seg.extracellular.e=0
cable.insert(h.xtra)
# Also get some 3d coordinates if stylized
h('define_shape()')

In [ ]:
# Set up the xtra mechanisms we added
h('load_file("interpxyz.hoc")')
h('load_file("setpointers.hoc")')

In [ ]:
# Set up the weights for just one sampling point
h('load_file("calcrxc.hoc")') # don't worry about division by zero (due to stylized axon expanding on the x axis while the default sampling point also lies there)
h(f'rho = {100/extracell_conductivity_mho_per_m }') # ohm cm
h('setelec(70000, 1000,0)') # following Brian's example
# print([seg.xtra.rx for seg in cable])

In [ ]:
# Add a current clamp
stim = h.IClamp(cable(0))
stim.delay = 50 # msec
stim.dur = 3 # msec
stim.amp = 1000 # nAmp

In [ ]:
# Set up recording
ers =  [
    h.Vector().record(cable(seg.x).xtra._ref_er)
    for seg in cable
]
recs = [
    h.Vector().record(cable(seg.x)._ref_v)
    for seg in cable
]
neuron_time = h.Vector().record(h._ref_t)

In [ ]:
# Prepare to run simulation
h.finitialize(-65 * neuron.units.mV)

In [ ]:
h.dt = 0.010 # msec
h.continuerun(153 * neuron.units.ms)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.subplot(211)
for i in range(10):
    plt.plot(neuron_time,recs[i*100])

In [ ]:
plt.plot(neuron_time,np.array(ers).sum(axis=0)); plt.xlim([49, 60]) # NOTE: er is in microvolts

In [ ]:
plt.plot(neuron_time,ers[20]);  plt.xlim([49,60])

In [ ]:
neuron_LFP_trace_V = np.array(ers).sum(axis=0)*1e-6 # microvolt to volt

Let's see how [Brian](https://brian2.readthedocs.io/en/stable/examples/compartmental.lfp.html) sees it: 

In [ ]:
from brian2 import *
defaultclock.dt = 0.01*ms
morpho = Cylinder(x=[0, 10]*cm, diameter=2*238*um, n=1000, type='axon')

El = 10.613* mV
ENa = 115*mV
EK = -12*mV
gl = 0.3*msiemens/cm**2
gNa0 = 120*msiemens/cm**2
gK = 36*msiemens/cm**2

# Typical equations
eqs = '''
# The same equations for the whole neuron, but possibly different parameter values
# distributed transmembrane current
Im = gl * (El-v) + gNa * m**3 * h * (ENa-v) + gK * n**4 * (EK-v) : amp/meter**2
I : amp (point current) # applied current
dm/dt = alpham * (1-m) - betam * m : 1
dn/dt = alphan * (1-n) - betan * n : 1
dh/dt = alphah * (1-h) - betah * h : 1
alpham = (0.1/mV) * 10*mV/exprel((-v+25*mV)/(10*mV))/ms : Hz
betam = 4 * exp(-v/(18*mV))/ms : Hz
alphah = 0.07 * exp(-v/(20*mV))/ms : Hz
betah = 1/(exp((-v+30*mV) / (10*mV)) + 1)/ms : Hz
alphan = (0.01/mV) * 10*mV/exprel((-v+10*mV)/(10*mV))/ms : Hz
betan = 0.125*exp(-v/(80*mV))/ms : Hz
gNa : siemens/meter**2
'''

neuron = SpatialNeuron(morphology=morpho, model=eqs, Cm=1*uF/cm**2,
                       Ri=35.4*ohm*cm, method="exponential_euler")
neuron.v = 0*mV
neuron.h = 1
neuron.m = 0
neuron.n = .5
neuron.I = 0
neuron.gNa = gNa0
neuron[5*cm:10*cm].gNa = 0*siemens/cm**2
M = StateMonitor(neuron, 'v', record=True)

# LFP recorder
Ne = 5 # Number of electrodes
sigma = extracell_conductivity_mho_per_m*siemens/meter # Resistivity of extracellular field (0.3-0.4 S/m)
lfp = NeuronGroup(Ne, model='''v : volt
                               x : meter
                               y : meter
                               z : meter''')
lfp.x = 7*cm # Off center (to be far from stimulating electrode)
lfp.y = [1*mm, 2*mm, 4*mm, 8*mm, 16*mm]
S = Synapses(neuron, lfp, model='''w : ohm*meter**2 (constant) # Weight in the LFP calculation
                                   v_post = w*(Ic_pre-Im_pre) : volt (summed)''')
S.summed_updaters['v_post'].when = 'after_groups'  # otherwise Ic has not yet been updated for the current time step.
S.connect()
S.w = 'area_pre/(4*pi*sigma)/((x_pre-x_post)**2+(y_pre-y_post)**2+(z_pre-z_post)**2)**.5'

Mlfp = StateMonitor(lfp, 'v', record=True)

run(50*ms, report='text')
neuron.I[0] = 1*uA  # current injection at one end
run(3*ms)
neuron.I = 0*amp
run(100*ms, report='text')

subplot(211)
for i in range(10):
    plot(M.t/ms, M.v[i*100]/mV)
ylabel('$V_m$ (mV)')
subplot(212)
for i in range(5):
    plot(M.t/ms, Mlfp.v[i]/mV)
ylabel('LFP (mV)')
xlabel('Time (ms)')
show()

In [ ]:
brian_LFP_trace_v = Mlfp.v[0]/volt

### Modelling the HH cable

Following the "Spatially detailed cells" chapter of the EDEN user guide

In [ ]:
%%writefile HH_Sodium.channel.nml
<neuroml>
<ionChannel id="na_chan" type="ionChannelHH" conductance="10pS" species="na">
    <gateHHrates id="m" instances="3">
        <forwardRate type="HHExpLinearRate" rate="1per_ms" midpoint="-40mV" scale="10mV"/>
        <reverseRate type="HHExpRate" rate="4per_ms" midpoint="-65mV" scale="-18mV"/>
    </gateHHrates>
    <gateHHrates id="h" instances="1">
        <forwardRate type="HHExpRate" rate="0.07per_ms" midpoint="-65mV" scale="-20mV"/>
        <reverseRate type="HHSigmoidRate" rate="1per_ms" midpoint="-35mV" scale="10mV"/>
    </gateHHrates>
</ionChannel>
</neuroml>

In [ ]:
%%writefile HH_Potassium.channel.nml
<neuroml>
<ionChannel id="k_chan" type="ionChannelHH" conductance="10pS" species="k">
    <gateHHrates id="n" instances="4">
        <forwardRate type="HHExpLinearRate" rate="0.1per_ms" midpoint="-55mV" scale="10mV"/>
        <reverseRate type="HHExpRate" rate="0.125per_ms" midpoint="-65mV" scale="-80mV"/>
    </gateHHrates>
</ionChannel>
</neuroml>

In [ ]:
%%writefile Passive_leak.channel.nml
<neuroml><ionChannelHH id="passiveChan"/></neuroml>

In [ ]:
cable_morph_file = f'''<neuroml>
<morphology id="Subdivided_Morph">
    <segment id="0" name="Axon_0">
        <proximal x="0" y="0" z="0" diameter="{cable_diameter_microns}"/>
        <distal x="{cable_length_microns/2}" y="0" z="0" diameter="{cable_diameter_microns}"/>
    </segment>
    <segment id="1" name="Axon_1">
        <parent id="0"/>
        <distal x="{cable_length_microns}" y="0" z="0" diameter="{cable_diameter_microns}"/>
    </segment>
    <segmentGroup id="firsthalf" neuroLexId="sao864921383"> 
        <property tag="numberInternalDivisions" value="{(cable_compartments+1)//2}"/>
        <member segment="0"/>
    </segmentGroup>
    <segmentGroup id="secondhalf" neuroLexId="sao864921383"> 
        <property tag="numberInternalDivisions" value="{(cable_compartments+0)//2}"/>
        <member segment="1"/>
    </segmentGroup>
</morphology>
</neuroml>'''
# print(cable_morph_file)
with open('Subdivided_Morph.nml', 'wt') as f: f.write(cable_morph_file)

### Modelling the biophysics distributions

Then, we'll apply the ion channels over the whole cable, with `erev`, `condDensity` and other biophysical parameters again as per the example in the NeuroML guide.

In [ ]:
%%writefile Subdivided_Cable.cell.nml
<neuroml>
<include href="Subdivided_Morph.nml"/>
<include href="HH_Sodium.channel.nml"/>
<include href="HH_Potassium.channel.nml"/>
<include href="Passive_leak.channel.nml"/>
<biophysicalProperties id="bioPhys_Passive">
    <membraneProperties>
        <channelDensity id="na_some" ionChannel="na_chan"     ion="na"           condDensity="0.12   S_per_cm2" erev="+50.0 mV" segmentGroup="firsthalf" />
        <channelDensity id="k_all"  ionChannel="k_chan"      ion="k"            condDensity="0.036  S_per_cm2" erev="-77.0 mV" /><!---->
        <channelDensity id="leak"   ionChannel="passiveChan" ion="non_specific" condDensity="0.0003 S_per_cm2" erev="-54.387 mV" />
        <spikeThresh value="0 mV"/>
        <specificCapacitance value="1.0 uF_per_cm2"/>
        <initMembPotential value="-65mV" />
    </membraneProperties>
    <intracellularProperties>
        <resistivity value="35.4 ohm_cm"/>   
    </intracellularProperties>
</biophysicalProperties>
<cell id="Subdivided_Cable" morphology="Subdivided_Morph" biophysicalProperties="bioPhys_Passive"/>
</neuroml>

### Modelling the experimental rig

We'll put a current clamp on the start of the cable and watch the spike wave and the LFP go.

In [ ]:
cable_pop_name='SingleAxon'
model_file = f'''<neuroml>
    <include href="Subdivided_Cable.cell.nml"/>
    <pulseGenerator id="pulseGen1" delay="50ms" duration="3ms" amplitude="1uA"/>
    <network id="Net">
        <population id="{cable_pop_name}" component="Subdivided_Cable" size="1" />
        <inputList id="stimInput_Subdivided_1" component="pulseGen1" population="{cable_pop_name}">
            <input id="0" target="../{cable_pop_name}[0]" segmentId="0" fractionAlong="0" destination="synapses"/>
        </inputList>
    </network>
</neuroml>'''
# print(model_file)
with open('Model_LonelyAxon.nml', 'wt') as f: f.write(model_file)

### Recording over the cell's full extent

We want to record the state of the cable throughout, over time.  We'll use EDEN's `explain_cell` and `GetLemsLocatorsForCell` [helper routines]( python_api.rst#module-eden_simulator.experimental ) to tell us which *spots on the neuron* to record, so that all *compartments* are covered.

<div class="alert alert-info">
Note

Because official NeuroML allows recording on `<segment>`s but only on the middle of each `<segment>` (when there's only one in our stylised morphology), we'll have to use [extended LEMS path]( extension_paths.rst#lems-paths-for-cell-locations ) syntax,  which is supported by EDEN but not official NeuroML.
</div>

In [ ]:
import eden_simulator
cells_info = eden_simulator.experimental.explain_cell('Model_LonelyAxon.nml')
cell_info = cells_info['Subdivided_Cable']

In [ ]:
comp_locators = eden_simulator.experimental.GetLemsLocatorsForCell(cell_info)
cell_voltage_paths = [ f'{cable_pop_name}[0]/{loc}/v' for loc in comp_locators ]
tabline = '\n        '
sim_file = f'''<Lems>
<Include href="Model_LonelyAxon.nml"/>
<Simulation id="MySim" length="153 ms" step="10 us" target="Net">
    <OutputFile id="MyOutFile" fileName="results.gen.txt">
        {tabline.join([ f'<OutputColumn id="v_0_{loc}"  quantity= "{path}"/>' for loc, path in zip(comp_locators, cell_voltage_paths)])}
    </OutputFile>
</Simulation>
<Target component="MySim"/></Lems>'''
# print(sim_file)
with open('Sim_LonelyAxon.xml', 'wt') as f: f.write(sim_file)

### Running the simulation and displaying results

In [ ]:
results = eden_simulator.runEden('Sim_LonelyAxon.xml',threads=1)
import numpy as np
rec_time_axis_sec = results['t']
neuron_waveforms = np.array([results[path] for path in cell_voltage_paths])

In [ ]:
plt.plot(M.t/second, M.v[0]/volt-0.065)
plt.plot(rec_time_axis_sec,neuron_waveforms[0],'--')
# plt.xlim([0.049,0.06])

Since we'll be doing physics with the simulation's results so that we can calculate the LFP, let's convert the $V_m$ data to a physical `Quantity` with the help of the `pint` [Python package]( https://pint.readthedocs.io/en/stable/user/defining-quantities.html ) for unit math.

In [ ]:
!pip install -q pint

In [ ]:
import numpy as np
from pint.registry import Quantity as Q
neuron_membrane_voltage = Q(neuron_waveforms,'V')
print("%d points in space, sampled over %d points in time." % neuron_membrane_voltage.shape)

### Calculating axial current

Just multiply membrane potential by the conductance matrix joining neurons. This matrix is so generally useful (see also the [imposed field](example_imposed_field.ipynb) example) that a python helper will soon provide it. <!-- TODO -->

In [ ]:
# First, get the axial resistance matrix: Ia = G*V. It follows the tree structure
comp_ga = cell_info['comp_conductance_to_parent'] # provided in nS
iii = []; jjj = []; vvv = [] # Construct the lists (i,j,v)
for i,p in enumerate(cell_info['comp_parent']):
    g = comp_ga[i]
    if p >= 0:
        iii += [ i, p, i, p] # add 4 sparse elms in one go
        jjj += [ p, i, i, p]
        vvv += [+g,+g,-g,-g] # same as:
        # axial_conductance_matrix[i,p] += g
        # axial_conductance_matrix[p,i] += g
        # axial_conductance_matrix[i,i] -= g
        # axial_conductance_matrix[p,p] -= g

import scipy
G = scipy.sparse.coo_matrix((vvv,(iii,jjj))).tocsr() # needs csr for some reason
G = Q(G,'nS')

axial_current = (G.m @ neuron_membrane_voltage.m)*G.units*neuron_membrane_voltage.units

### Calculating injected current

As explained previously, we'll have to figure that out from our model for the time being.  Oh well...

In [ ]:
comp_Iclamp = Q(np.zeros_like(neuron_membrane_voltage), 'pA')
# facts of life for this model
comp_Iclamp[0,(50e-3 < rec_time_axis_sec)&(rec_time_axis_sec <= 53e-3+11e-6)] = Q(1,'uA')
# NOTE: Check the inequalities.
# The effect of Im will appear on the timestep AFTER it nominally starts and ends,
#  as the capacitive and axial currents can't have reacted to it before it started!
# Might as well apply a highpass or median filter since the simulation timestep should be less than 25 μs.

And check that the total membrane current makes sense on, say, the soma. 

In [ ]:
neuron_membrane_current = -(-comp_Iclamp + -axial_current) # going outward!
comp_capa = Q(cell_info['comp_capacitance'], 'pF')
capa_current = comp_capa[:,None] * np.diff(neuron_membrane_voltage, axis=-1)/Q(np.diff(timevec),'s') # outward! 
ion_current = (neuron_membrane_current[:,1:]+neuron_membrane_current[:,:-1])/2 - capa_current # outward! 
timevec = rec_time_axis_sec
comp = 100; xlim = [0.049,0.056]
comp = 490; xlim = [0.055,0.060]
# comp = 600; xlim = [0.055,0.060]
fig, (ax1, ax2) = plt.subplots(2,1,figsize=(12,6), dpi=200)
ax1.plot(timevec,-comp_Iclamp[comp].m_as('uA'), '-',lw=1,label='Clamp current');
ax1.plot(timevec,-axial_current[comp].m_as('uA'), '-',lw=1,label='Axial current');
ax1.plot((timevec[1:]+timevec[:-1])/2,capa_current[comp].m_as('uA'), '-',lw=1, label='Capacitive');
ax1.plot((timevec[1:]+timevec[:-1])/2,ion_current[comp].m_as('uA'), '--',lw=1, label='Ion+Leak (deduced)');
ax1.plot(timevec,neuron_membrane_current[comp].m_as('uA'),'k--',lw=1,label='Total Im');
ax1.set_ylabel('uA'); ax1.legend()
ax1.set_xlim(xlim)
ax2.plot(timevec,neuron_membrane_voltage[comp].m_as('mV')); ax2.set_ylabel('Vm (mV)'); ax2.set_xlabel('Time (sec)')
ax2.set_xlim(xlim);

In [ ]:
import scipy
# Icap + Iion + Iax + Iclamp = 0 (outward); Im = Icap + Iion = -Iax -Iclamp (outward)
# Iion = Im - Icap 
plt.figure(figsize=(12,6), dpi=200)
plt.plot((timevec[1:]+timevec[:-1])/2,capa_current[0].m_as('uA'));
plt.plot((timevec[1:]+timevec[:-1])/2,scipy.signal.medfilt((-capa_current +neuron_membrane_current[:,:-1])[0].m_as('uA'),[3]));
plt.xlim([0.049, 0.056]);

In [ ]:
import matplotlib
plt.subplots(figsize=(22,16), dpi=100)
plt.imshow(neuron_membrane_current.m_as('uA')[:,5000:6200],aspect='auto',interpolation='None',norm=matplotlib.colors.CenteredNorm(),cmap='bwr');plt.colorbar(); #plt.xlim([4900,5400])

In [ ]:
plt.plot(neuron_membrane_current.m_as('uA').sum(axis=0))

## Post-process $I_m$ to get the LFP

The other half of calculating the LFP is how much each point outside the cell is being affected by the current flow on each part of the neuron (that is, mostly by the closest compartments, but also slightly from the more distant ones).

The usual assumptions apply for this example: The extracellular medium is uniform, highly conductive, isotropic and unobstructed until far away.  If you need a more sophisticated model, apply the relevant adjustments to the following.  (Perhaps a separate simulation process just for the LFP is called for, see the other articles TBD on "Pipelines" generally.)
<!-- LATER pipelines -->

Just for illustration, we'll use one element per compartment. It would be even better if each NeuroML `<segment>` is applied separately, if it's not too much for the computer.  
Alternatively, the [LFPykit]( https://github.com/LFPy/LFPykit ) could also be used to calculate the coupling matrix.

In [ ]:
extracell_conductivity = Q(extracell_conductivity_mho_per_m, 'S/m')

source_points = cell_info['comp_midpoint']
x_space = [70000]; y_space = [1000,2000,4000,8000,16000]; z_space = [0]
x, y, z = np.meshgrid(x_space, y_space, z_space, indexing='ij')
sampling_points = np.stack((x,y,z),axis=-1).reshape((-1,3))
print("Sampling points:", sampling_points)

def GetLfpCoupling(sampling_points,source_points,extracell_conductivity):
    # Vectorize calculation of pairwise distance matrix https://jaykmody.com/blog/distance-matrices-with-numpy/
    distmat =  source_points @ sampling_points.T
    distmat = np.sum(sampling_points**2,axis=-1) - 2*distmat + np.sum(source_points**2,axis=-1)[:,None]
    distmat = np.sqrt(distmat)
    distmat = Q(distmat, 'um')
    
    # LATER: Exclude current from points inside the cell, thereby avoiding singularities as a bonus.
    coupling = (1 / (4*np.pi*extracell_conductivity*distmat)).to('uV/pA')
    # NB: Coupling is positive *although* potential *drops* along the electric field's lines! Think of it this way:
    #     Since potential is 0 at infinity, the potential any closer to the source has to be positive! Thus I/4πσr
    print('Coupling is calculated for %d current sources times %d sampling points.' % coupling.shape)
    return coupling

def QMatmul(A,v):# TODO subclass quantity to allow sparse matmul i guess?
    import operator
    return A._mul_div(v, operator.matmul, operator.mul)
def get_lfp(coupling,membrane_current): return QMatmul(coupling, membrane_current).to('uV')

coupling = GetLfpCoupling(sampling_points,source_points,extracell_conductivity)
lfp_uvolt = get_lfp(coupling,neuron_membrane_current).m_as('uV').T

## Visualisation

In [ ]:
lines_neuron = plt.plot(neuron_time/1e3,neuron_LFP_trace_V*1e3,'k',label="Neuron")
lines_eden   = plt.plot(timevec,lfp_uvolt/1e3,'-', label=["Eden"]+[None]*4)
lines_brian  = plt.plot(M.t/second, Mlfp.v.T/mV, '--',label=['Brian']+[None]*4)
plt.xlim([0.049, 0.063]); plt.legend()

## Check

In [ ]:
# Compare Eden vs Neuron, Brian has neglected the monopolar effect of the current clamp because it wrongly includes it into the Im.
raise NotImplementedError()

## Now for the extracellular stim test

### What does Neuron say

In [ ]:
stim.amp = 0 # Remove the current clamp

In [ ]:
extracell_source_current_mA = -0.050
extracell_source_delay_ms = 1
extracell_source_duration_ms = 6+11e-3
h('load_file("stim.hoc")')
h(f'setstim({extracell_source_delay_ms},{extracell_source_duration_ms},{extracell_source_current_mA})'); # following Brian's example

In [ ]:
h.finitialize(-65 * neuron.units.mV)
h.continuerun(13 * neuron.units.ms);

In [ ]:
import matplotlib.pyplot as plt
for i in range(10):
    plt.plot(neuron_time,recs[i*100],alpha=1,label=f'{i}')
plt.legend();

In [ ]:
neuron_extLFP_trace_V = np.array(ers).sum(axis=0)*1e-6 # microvolt to volt
plt.plot(neuron_time,neuron_extLFP_trace_V);

### Calculate equivalent currents

In [ ]:
import numpy as np
def ElectricPotential_PointSource(x,y,z, loc=[0,0,0], extracell_conductivity=None, source_current=None):
    "Get the electric potential at the selected locations and a point-current source, for x,y,z in microns."
    distance_um = np.linalg.norm(np.array([x,y,z]).T - loc, axis=-1)
    # print(distance_um.shape)
    distance = Q(distance_um, 'um')
    return (source_current / (4*np.pi*extracell_conductivity*distance))

Vext = ElectricPotential_PointSource(*cell_info['comp_midpoint'].T,loc=[70000,1000,0],
        extracell_conductivity=extracell_conductivity, source_current=Q(extracell_source_current_mA,'mA'))
Iext = QMatmul(G,Vext)

In [ ]:
rxs = [seg.xtra.rx for seg in cable] # in megohms
plt.plot((Vext/Q(extracell_source_current_mA,'mA')).m_as('MΩ') - rxs)

### Run it on Eden

In [ ]:
comp_locators = eden_simulator.experimental.GetLemsLocatorsForCell(cell_info)
cell_voltage_paths = [ f'{cable_pop_name}[0]/{loc}/v' for loc in comp_locators ]
tabline = '\n        '
sim_file = f'''<neuroml>
<include href="Subdivided_Cable.cell.nml"/>
<pulseGenerator id="pulseGen1" delay="{extracell_source_delay_ms}ms" duration="{extracell_source_duration_ms}ms" amplitude="1nA"/>
<network id="Net">
    <population id="{cable_pop_name}" component="Subdivided_Cable" size="1" />
    <inputList id="stimInput_Subdivided_1" component="pulseGen1" population="{cable_pop_name}">
        {tabline.join([ f'<inputW id="{comp}" target="{cable_pop_name}[0]" weight="{curr}"'
           +f' segmentId="{cell_info["comp_midpoint_segment"][comp]}"'
           +f' fractionAlong="{cell_info["comp_midpoint_fractionAlong"][comp]}" destination="synapses"/>'
           for comp, curr in enumerate(Iext.m_as('nA'))])}
    </inputList>
</network>
<Simulation id="MySim" length="13 ms" step="10 us" target="Net">
    <OutputFile id="MyOutFile" fileName="results.gen.txt">
        {tabline.join([ f'<OutputColumn id="v_0_{loc}"  quantity= "{path}"/>' for loc, path in zip(comp_locators, cell_voltage_paths)])}
    </OutputFile>
</Simulation>
<Target component="MySim"/></neuroml>'''
# print(sim_file)
with open('Sim_LonelyAxonField.xml', 'wt') as f: f.write(sim_file)

In [ ]:
results = eden_simulator.runEden('Sim_LonelyAxonField.xml',threads=1)
rec_time_axis_sec = results['t']
neuron_waveforms = np.array([results[path] for path in cell_voltage_paths])

**NB:** Watch out when both imposing a field, *and* geting the induced LFP from the axial + clamp current!  
Either offset by Vext when calculating axial current, or alternatively include the pseudo-current in the patch-clamp total. (Both add up to the same true axial-under-voltage-shift current.)

In [ ]:
neuron_membrane_voltage = Q(neuron_waveforms,'V')
axial_current = QMatmul(G,neuron_membrane_voltage)
comp_Iclamp = Q(np.zeros_like(neuron_membrane_voltage), 'pA')
comp_Iclamp[:,(extracell_source_delay_ms/1000 < rec_time_axis_sec)&(rec_time_axis_sec <= (extracell_source_delay_ms+extracell_source_duration_ms)/1000+11e-6)] = Iext[:,None]
neuron_membrane_current = -(-comp_Iclamp + -axial_current) # going outward!

In [ ]:
coupling = GetLfpCoupling(np.array([[70000,1000,0]]),source_points,extracell_conductivity)
lfp_uvolt = get_lfp(coupling,neuron_membrane_current).m_as('uV').T

In [ ]:
plt.plot(coupling.m_as('MΩ').squeeze() - rxs)

In [ ]:
plt.plot((neuron_membrane_current[700]*coupling[700]).m_as('uV'))
plt.plot(ers[700])

### Check

In [ ]:
for i in range(10):
    plt.plot(rec_time_axis_sec,neuron_waveforms[i*100],alpha=.5)
for i in range(10):
    plt.plot(neuron_time/1e3,recs[i*100]/1e3,'--',label=f'{i}')
plt.legend();

In [ ]:
plt.imshow(neuron_waveforms - np.array(recs)/1e3); plt.colorbar()

In [ ]:
plt.plot((lfp_uvolt.T[700]/1e6 - neuron_extLFP_trace_V))

In [ ]:
lines_eden   = plt.plot(rec_time_axis_sec,lfp_uvolt/1e3,'-', label="Eden")
lines_neuron = plt.plot(neuron_time/1e3,neuron_extLFP_trace_V*1e3,'k--',label="Neuron"); plt.legend();

In [ ]:
# Compare Eden vs Neuron
raise NotImplementedError()